In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from tqdm import tqdm
import json
import re

import matplotlib.pyplot as plt
import matplotlib.colors as mcl
import seaborn as sns

from IPython.display import HTML

import uuid

import networkx as nx
from matplotlib.lines import Line2D
from IPython.display import clear_output

In [3]:
species = "megalopta"
datastack = "PB1"

if species == "megalopta":
    if datastack == 'FBEB':
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_EPG.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_PEN.csv', dtype={'Root ID': str})
        er = pd.read_csv('../syntables/updated_google_sheets/Megalopta_EB_preprint_FINAL - FBEB_ER.csv', dtype={'Root ID': str})
        #er = er[er['Completed']==True]
        nametable = pd.concat((epg, pen, er))
    elif datastack == 'PB1':
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_EPG.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_PEN.csv', dtype={'Root ID': str}) 
        d7 = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB_preprint_FINAL - PB1_delta7.csv', dtype={'Root ID': str})
        nametable = pd.concat((epg, pen, d7))
    elif datastack == 'PB2':
        d7 = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_delta7.csv', dtype={'Root ID': str})
        epg = pd.read_csv('../syntables/updated_google_sheets/Megalopta_PB2_preprint_FINAL - PB2_EPG.csv', dtype={'Root ID': str})
        nametable = pd.concat((d7, epg))
    elif datastack == 'NO':
        lno = pd.read_csv('../syntables/updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_LNO.csv', dtype={'Root ID': str})
        pen = pd.read_csv('../syntables/updated_google_sheets/Megalopta_NOr neuron_CAVE progress - NOr_PEN.csv', dtype={'Root ID': str})
        nametable = pd.concat((lno, pen))
    
elif species == "eciton":
    d7 = pd.read_csv('../syntables/updated_google_sheets/Eciton_PB neuron_CAVE progress - eciton_PB_Delta7.csv', dtype={'Root ID': str})
    epg = pd.read_csv('../syntables/updated_google_sheets/Eciton_PB neuron_CAVE progress - eciton_PB_EPG.csv', dtype={'Root ID': str})
    nametable_pb = pd.concat((epg, d7))
    nametable_pb["roi"] = "PB"

    nametable = nametable_pb.copy()

nametable = nametable[['Catmaid name', 'Root ID', 'CATMAID skid']].rename(columns={'Catmaid name':'name', 'Root ID':'root_ids', 'CATMAID skid':'skid'})

nametable['skid'] = nametable['skid'].fillna(0).astype(float).astype(int)

# make sure to fill na
nametable['root_ids'] = nametable['root_ids'].fillna('')

# convert each cell in 'root_ids' from a comma-separated string to a list of integers, ignoring empty strings
nametable['root_ids'] = nametable['root_ids'].apply(lambda x: [int(seg) for seg in x.split(',') if seg.strip()])

# remove empty values
nametable = nametable[nametable['root_ids'].apply(lambda x: len(x) > 0)].copy()

'''NOTE this removes any neuron with skid == 0. This is to remove unproofread cells...if you 
want to keep these neurons, you have to add a skid otherwise the update will not work properly.'''
nametable = nametable[nametable['skid'] != 0]

nametable

,name,root_ids,skid
0,EPG_L2_55944,[576460752521344750],55944
1,EPG_L2_55952,[576460752510496133],55952
2,EPG_L3_54939,[576460752494001689],54939
3,EPG_L3_57518,[576460752497676789],57518
4,EPG_L3_57529,[576460752460921048],57529
...,...,...,...
37,delta7_R_L2L10R7_83213,[576460752550908713],83213
38,delta7_R_L2L10R7_83722,[576460752608580253],83722
39,delta7_R_L2L10R7_84178,[576460752490172734],84178
40,delta7_R_L2L10R7_84473,[576460752543874747],84473


In [ ]:
d7 = nametable[nametable["name"].str.contains(r"L1L9", na=False)]
d = d7['skid'].to_list()
d

In [9]:
import json
import os
from copy import deepcopy

In [37]:
species = "eciton"
input_dir = "./json_to_update"
output_dir = "./updated_json"

json_filenames = ["L1L9", "L2L10", "L3R6", "L4R5", "L5R4", "L6R3", "L7R2", "L8R1"]
group_colours = ["#575D8E", "#A64DBD", "#C960E6", "#C33889", "#D93B4B", "#F28C09", "#68B750", "#3A8B72", "#4FAABF"]

if species == "megalopta":
    TOP_LEVEL_REPLACEMENTS = {
        "dimensions": {
            "x": [1e-8, "m"],
            "y": [1e-8, "m"],
            "z": [5e-8, "m"]
        },
        "position": [
            27053.529296875,
            5470.01220703125,
            2047.4898681640625
        ],
        "crossSectionScale": 3.037731956345071,
        "projectionOrientation": [
            0.8656170964241028,
            -0.043054498732089996,
            -0.05414479970932007,
            0.49590495228767395
        ],
        "projectionScale": 36328.46381081182,
        "crossSectionBackgroundColor": "#ffffff",
        "projectionBackgroundColor": "#ffffff"
    }
    neuropil_segment = ["2"]
    neuropil_q = "2"
elif species == "eciton":
    TOP_LEVEL_REPLACEMENTS = {
        "dimensions": {
            "x": [1e-8, "m"],
            "y": [1e-8, "m"],
            "z": [5e-8, "m"]
        },
      "position": [
        17904.76171875,
        25679.083984375,
        2149.813720703125
      ],
      "crossSectionScale": 0.5737534122933909,
      "projectionOrientation": [
        -0.04075660556554794,
        0.6548686027526855,
        0.7545750737190247,
        -0.010122704319655895
      ],
      "projectionScale": 22730.59585032473,
      "crossSectionBackgroundColor": "#ffffff",
      "projectionBackgroundColor": "#ffffff"
    }
    neuropil_segment = ["0"]
    neuropil_q = "0"

REQUIRED_LAST_LAYER = {
    "type": "segmentation",
    "source": f"gs://heinze-lab/catmaid_meshes/{species}/",
    "tab": "segments",
    "objectAlpha": 0.06,
    "segments": neuropil_segment,
    "segmentQuery": neuropil_q,
    "colorSeed": 3244830578,
    "segmentDefaultColor": "#cccccc",
    "name": f"{species}"
}

# WHITE_BACKGROUND = { # to check and add if not already present
#   "crossSectionBackgroundColor": "#ffffff",
#   "projectionBackgroundColor": "#ffffff"
# }


def layer_contains_text(obj, needles):
    if isinstance(obj, dict):
        return any(layer_contains_text(v, needles) for v in obj.values())
    if isinstance(obj, list):
        return any(layer_contains_text(v, needles) for v in obj)
    if isinstance(obj, str):
        s = obj.lower()
        return any(n.lower() in s for n in needles)
    return False


def get_segments_for_coloring(layer):
    """
    Use the actual 'segments' list for coloring.
    Exclude negated entries like '!2'.
    """
    segments = layer.get("segments", [])
    if not isinstance(segments, list):
        return []

    cleaned = []
    for s in segments:
        s = str(s).strip()
        if not s:
            continue
        if s.startswith("!"):
            continue
        cleaned.append(s)
    return cleaned


def build_segment_colors(segment_ids, color):
    return {seg_id: color.lower() for seg_id in segment_ids}


def is_catmaid_mesh_layer(layer):
    """
    Skip the neuropil mesh layer.
    """
    source = layer.get("source")

    if isinstance(source, str):
        return "catmaid_meshes" in source.lower()

    if isinstance(source, dict):
        url = source.get("url", "")
        if isinstance(url, str) and "catmaid_meshes" in url.lower():
            return True

    return False


def is_catmaid_skeleton_layer(layer):
    """
    Detect skeleton layers that need skeletonRendering.
    """
    source = layer.get("source")

    if isinstance(source, str):
        return "catmaid-skeletons" in source.lower()

    if isinstance(source, dict):
        url = source.get("url", "")
        if isinstance(url, str) and "catmaid-skeletons" in url.lower():
            return True

    return False


def required_last_layer_exists(layers):
    for layer in layers:
        if not isinstance(layer, dict):
            continue

        if layer.get("name") != REQUIRED_LAST_LAYER["name"]:
            continue

        source = layer.get("source")
        if source == REQUIRED_LAST_LAYER["source"]:
            return True

    return False


def process_layers(layers, group_color):
    for layer in layers:
        if not isinstance(layer, dict):
            continue

        # Add skeletonRendering to catmaid skeleton layers
        if is_catmaid_skeleton_layer(layer):
            layer["skeletonRendering"] = {"lineWidth3d": 7}

        # Add / overwrite segmentColors for non-catmaid_meshes layers
        if not is_catmaid_mesh_layer(layer):
            segment_ids = get_segments_for_coloring(layer)
            if segment_ids:
                layer["segmentColors"] = build_segment_colors(segment_ids, group_color)

    return layers


def update_json(data, group_color):
    data = deepcopy(data)

    # Replace top-level values
    for key, value in TOP_LEVEL_REPLACEMENTS.items():
        data[key] = deepcopy(value)

    layers = data.get("layers", [])
    if not isinstance(layers, list):
        raise ValueError("'layers' is not a list")

    data["layers"] = process_layers(layers, group_color)

    # Add required last layer if missing
    if not required_last_layer_exists(data["layers"]):
        data["layers"].append(deepcopy(REQUIRED_LAST_LAYER))

    return data

In [38]:
os.makedirs(output_dir, exist_ok=True)

for i, base_name in enumerate(json_filenames):
    input_path = os.path.join(input_dir, f"{base_name}.json")
    output_path = os.path.join(output_dir, f"{base_name}.json")
    color = group_colours[i]

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    updated = update_json(data, color)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(updated, f, indent=2)

    print(f"Updated {base_name} -> {output_path}")

Updated L1L9 -> ./updated_json\L1L9.json
Updated L2L10 -> ./updated_json\L2L10.json
Updated L3R6 -> ./updated_json\L3R6.json
Updated L4R5 -> ./updated_json\L4R5.json
Updated L5R4 -> ./updated_json\L5R4.json
Updated L6R3 -> ./updated_json\L6R3.json
Updated L7R2 -> ./updated_json\L7R2.json
Updated L8R1 -> ./updated_json\L8R1.json


In [32]:
file_path = "./updated_json/L1L9.json"
with open(file_path, 'r') as file:
    js = json.load(file)

js

{'dimensions': {'x': [1e-08, 'm'], 'y': [1e-08, 'm'], 'z': [5e-08, 'm']},
 'position': [23471.265625, 26332.52734375, 1820.4281005859375],
 'crossSectionScale': 0.5737534122933909,
 'projectionOrientation': [-0.03984975814819336,
  0.655428946018219,
  0.7541475296020508,
  -0.009298511780798435],
 'projectionScale': 30428.465299304033,
 'layers': [{'type': 'image',
   'source': 'precomputed://https://lweb1569.srv.lu.se/catmaid/data/eciton/Eciton_PB_1026',
   'tab': 'source',
   'name': 'imagery'},
  {'type': 'segmentation',
   'source': 'graphene://middleauth+https://local.cave.braininbrain.org/segmentation/table/heinze_eciton_PB_v1',
   'tab': 'segments',
   'segments': ['576460752490111867',
    '576460752529758329',
    '576460752441507663',
    '576460752484838471',
    '576460752496147085'],
   'name': 'segmentation',
   'segmentColors': {'576460752490111867': '#575d8e',
    '576460752529758329': '#575d8e',
    '576460752441507663': '#575d8e',
    '576460752484838471': '#575d8e',